# SHIPP — Who writes the Silver tables?

Delta keeps a history of every write, including the notebook or job that did it.
Rule: each Silver table must have **exactly one** writer
(`10_silver_listings_dev` for listings, `11_silver_requests_dev` for requests).

If you see a second `notebook_id`, open the printed link, find the `saveAsTable` / `write`
that targets this table, and delete it (or switch it to read-only).

In [ ]:
from pyspark.sql import functions as F

SILVER_TABLES = [
    "bootcamp_students.shipp_silver.silver_listings",
    "bootcamp_students.shipp_silver.silver_requests",
]

try:
    WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    WORKSPACE_URL = None

for table in SILVER_TABLES:
    print("=" * 80)
    print(table)
    try:
        history = spark.sql(f"DESCRIBE HISTORY {table}")
    except Exception as e:
        print("  not readable:", type(e).__name__, str(e)[:200])
        continue

    display(
        history.select(
            "version", "timestamp", "userName", "operation",
            F.col("notebook")["notebookId"].alias("notebook_id"),
            F.col("job")["jobId"].alias("job_id"),
            F.col("operationMetrics")["numOutputRows"].alias("rows_written"),
        ).limit(30)
    )

    writers = (
        history
        .filter(~F.col("operation").isin("OPTIMIZE", "VACUUM START", "VACUUM END",
                                         "SET TBLPROPERTIES", "ANALYZE", "RESTORE"))
        .select(F.col("notebook")["notebookId"].alias("notebook_id"),
                F.col("job")["jobId"].alias("job_id"))
        .distinct()
        .collect()
    )
    print(f"Distinct writers: {len(writers)}")
    for w in writers:
        link = (f"https://{WORKSPACE_URL}/editor/notebooks/{w['notebook_id']}"
                if WORKSPACE_URL and w["notebook_id"] else "")
        print(f"  notebook_id={w['notebook_id']}  job_id={w['job_id']}  {link}")